# Word2Vec Pipeline

This notebook covers data loading, cleaning/preprocessing, Word2Vec feature extraction
(average word vectors per document), and model training/comparison for the fake news
dataset (`title`, `text`, `label`).

Reusable functions from `utils.py` are used.

## 1. Setup

In [1]:
import os
os.chdir('../')
print(os.getcwd())

c:\Users\kriti\Dropbox\PC\Desktop\Ironhack_AI_Engineering\Projects\Project_2\Project-2-Natural-Language-Processing


In [2]:
# Core imports
import utils

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC


## 2. Load Data

In [3]:
raw_data = pd.read_csv('dataset/data.csv')
print("Raw shape:", raw_data.shape)
# subject/date dropped here -- see EDA notebook for the leakage check behind this decision
raw_data = raw_data.drop(columns=['subject', 'date'])
print("Shape after dropping subject/date:", raw_data.shape)
raw_data.head()

Raw shape: (39942, 5)
Shape after dropping subject/date: (39942, 3)


,label,title,text
0,1,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...
1,1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...
2,1,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...
3,1,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...
4,1,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...


## 3. Train / Test Split

In [4]:
TEXT_COLUMNS = ['title', 'text']

X = raw_data.drop(columns='label')
y = raw_data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (31953, 2)
Test shape: (7989, 2)


## 4. Text Cleaning & Preprocessing

In [ ]:
# # Preprocess text columns and keep only preprocessed columns
# X_train_clean = utils.preprocess_text_columns(X_train, TEXT_COLUMNS)[[f'{c}_clean' for c in TEXT_COLUMNS]]
# X_test_clean = utils.preprocess_text_columns(X_test, TEXT_COLUMNS)[[f'{c}_clean' for c in TEXT_COLUMNS]]

# X_train_clean.columns = TEXT_COLUMNS
# X_test_clean.columns = TEXT_COLUMNS

# X_train_clean.to_csv('X_train_clean.csv')
# X_test_clean.to_csv('X_test_clean.csv')

# X_train_clean.head()


: 

### Reload cached cleaned text (skip re-running the cell above once cached)

In [5]:
X_train_clean = pd.read_csv('X_train_clean.csv', index_col=0)
X_test_clean = pd.read_csv('X_test_clean.csv', index_col=0)

print(X_train_clean.isna().sum())
print(X_test_clean.isna().sum())

title      5
text     549
dtype: int64
title      2
text     144
dtype: int64


### Fill null values with empty string

After cleaning/preprocessing, some rows end up empty (e.g. originally empty `text`,
or text that became empty after stripping HTML/special characters). Filling with `''`
keeps these rows usable for vectorization instead of dropping them.

In [6]:
X_train_clean = X_train_clean.fillna('')
X_test_clean = X_test_clean.fillna('')

print("Remaining NaNs:", X_train_clean.isna().sum().sum(), X_test_clean.isna().sum().sum())

Remaining NaNs: 0 0


## 5. Word2Vec Feature Extraction

Train one Word2Vec model per text column **on the train tokens only**, then represent
each document as the **average of its word vectors** (a simple, common baseline for
turning Word2Vec into fixed-size document features). Combine columns with `np.hstack`
(dense arrays, since averaged Word2Vec vectors aren't sparse).

In [7]:
def get_avg_vector(text, model, dim=100):
    """Average the Word2Vec vectors of all known words in a document.
    Falls back to a zero vector if none of the words are in the vocabulary."""
    words = text.split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(dim)


In [8]:
w2v_vectorizers = {}
X_train_text_features = []
X_test_text_features = []

for col in TEXT_COLUMNS:
    train_tokens = X_train_clean[col].apply(lambda t: t.split())
    w2v_model = Word2Vec(sentences=train_tokens, vector_size=100, window=5, min_count=2, sg=1)

    train_vec = np.vstack(X_train_clean[col].apply(lambda t: get_avg_vector(t, w2v_model)))
    test_vec = np.vstack(X_test_clean[col].apply(lambda t: get_avg_vector(t, w2v_model)))

    w2v_vectorizers[col] = w2v_model
    X_train_text_features.append(train_vec)
    X_test_text_features.append(test_vec)

    print(f"{col}: train shape {train_vec.shape}, test shape {test_vec.shape}")


title: train shape (31953, 100), test shape (7989, 100)
text: train shape (31953, 100), test shape (7989, 100)


In [9]:
# --- Combine all Word2Vec features into one matrix (dense) ---
X_train_final = np.hstack(X_train_text_features)
X_test_final = np.hstack(X_test_text_features)

print("Final train feature matrix:", X_train_final.shape)
print("Final test feature matrix:", X_test_final.shape)

Final train feature matrix: (31953, 200)
Final test feature matrix: (7989, 200)


## 6. Model Training

Using `utils.train_evaluate_model()` — trains, evaluates (train/test accuracy, precision,
recall, F1, confusion matrix), prints a summary, saves the result to `results_log.jsonl`,
and pickles the fitted model to `saved_models/`, all in one call.

Note: `MultinomialNB` not used as it requires non-negative features, but averaged Word2Vec vectors can be
negative


In [10]:
# Container to collect results from every model
RESULTS = []

In [11]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_results = utils.train_evaluate_model(
    lr_model, 'Word2Vec', 'Logistic Regression - Word2Vec',
    X_train_final, y_train, X_test_final, y_test,
    comments="Baseline LR model with word2vec(100 dims, sg=1)",
    results_container=RESULTS
)

--- Logistic Regression - Word2Vec | Word2Vec (Baseline LR model with word2vec(100 dims, sg=1)) ---
Train Accuracy: 0.9786
Test Accuracy:  0.9778
Precision:      0.9787
Recall:         0.9770
F1 Score:       0.9779
Confusion Matrix:
 [[3904   85]
 [  92 3908]]
Model saved to: models\Logistic_Regression_-_Word2Vec_Word2Vec_2026-07-10_12-27-17.pkl



In [12]:
svm_model = LinearSVC()
svm_results = utils.train_evaluate_model(
    svm_model, 'Word2Vec', 'SVM - Word2Vec',
    X_train_final, y_train, X_test_final, y_test,
    comments="SVM model with word2vec(100 dims, sg=1)",
    results_container=RESULTS
)

--- SVM - Word2Vec | Word2Vec (SVM model with word2vec(100 dims, sg=1)) ---
Train Accuracy: 0.9842
Test Accuracy:  0.9832
Precision:      0.9840
Recall:         0.9825
F1 Score:       0.9832
Confusion Matrix:
 [[3925   64]
 [  70 3930]]
Model saved to: models\SVM_-_Word2Vec_Word2Vec_2026-07-10_12-27-20.pkl



In [13]:
rf_model = RandomForestClassifier(random_state=42)
rf_results = utils.train_evaluate_model(
    rf_model, 'Word2Vec', 'Random Forest - Word2Vec',
    X_train_final, y_train, X_test_final, y_test,
    comments="Random Forest model with word2vec(100 dims, sg=1)",
    results_container=RESULTS
) # # 1m

--- Random Forest - Word2Vec | Word2Vec (Random Forest model with word2vec(100 dims, sg=1)) ---
Train Accuracy: 1.0000
Test Accuracy:  0.9609
Precision:      0.9706
Recall:         0.9507
F1 Score:       0.9606
Confusion Matrix:
 [[3874  115]
 [ 197 3803]]
Model saved to: models\Random_Forest_-_Word2Vec_Word2Vec_2026-07-10_12-27-24.pkl



In [ ]:
l

## 7. Model Comparison & Selection

Results are reloaded from `results_log.jsonl` (rather than the in-memory `RESULTS` list),
so this section also picks up results from any earlier session/kernel restart — including
results from the TF-IDF and BoW notebooks, since they share the same log file.

In [14]:
results_df = utils.load_results_from_file()
results_df = results_df.sort_values('f1_score', ascending=False).reset_index(drop=True)
results_df


,timestamp,model_name,method,comments,train_accuracy,test_accuracy,precision,recall,f1_score,confusion_matrix,model_path
0,2026-07-10 12:19:35,Random Forest - BoW,BoW,"Random Forest model with bow(5000 features, ng...",1.000000,0.998623,0.998002,0.99925,0.998626,"[[3981, 8], [3, 3997]]",models\Random_Forest_-_BoW_BoW_2026-07-10_12-1...
1,2026-07-10 12:07:35,Random Forest - TF-IDF,TF-IDF,"Random Forest model with tfidf(5000 features, ...",1.000000,0.998248,0.997504,0.99900,0.998251,"[[3979, 10], [4, 3996]]",models\Random_Forest_-_TF-IDF_TF-IDF_2026-07-1...
2,2026-07-10 12:19:32,Logistic Regression - BoW,BoW,"Baseline LR model with bow(5000 features, ngra...",0.999969,0.996620,0.996749,0.99650,0.996625,"[[3976, 13], [14, 3986]]",models\Logistic_Regression_-_BoW_BoW_2026-07-1...
3,2026-07-10 12:19:33,SVM - BoW,BoW,"SVM model with bow(5000 features, ngram_range=...",0.999969,0.996245,0.995754,0.99675,0.996252,"[[3972, 17], [13, 3987]]",models\SVM_-_BoW_BoW_2026-07-10_12-19-33.pkl
4,2026-07-10 12:07:34,SVM - TF-IDF,TF-IDF,"SVM model with tfidf(5000 features, ngram_rang...",0.999969,0.995744,0.995255,0.99625,0.995752,"[[3970, 19], [15, 3985]]",models\SVM_-_TF-IDF_TF-IDF_2026-07-10_12-07-34...
5,2026-07-10 12:07:33,Logistic Regression - TF-IDF,TF-IDF,"Baseline LR model with tfidf(5000 features, ng...",0.996839,0.991864,0.990771,0.99300,0.991884,"[[3952, 37], [28, 3972]]",models\Logistic_Regression_-_TF-IDF_TF-IDF_202...
6,2026-07-10 12:27:20,SVM - Word2Vec,Word2Vec,"SVM model with word2vec(100 dims, sg=1)",0.984227,0.983227,0.983976,0.98250,0.983237,"[[3925, 64], [70, 3930]]",models\SVM_-_Word2Vec_Word2Vec_2026-07-10_12-2...
7,2026-07-10 12:27:17,Logistic Regression - Word2Vec,Word2Vec,"Baseline LR model with word2vec(100 dims, sg=1)",0.978625,0.977845,0.978713,0.97700,0.977856,"[[3904, 85], [92, 3908]]",models\Logistic_Regression_-_Word2Vec_Word2Vec...
8,2026-07-10 12:19:33,Naive Bayes - BoW,BoW,"Naive Bayes model with bow(5000 features, ngra...",0.964542,0.960446,0.957072,0.96425,0.960648,"[[3816, 173], [143, 3857]]",models\Naive_Bayes_-_BoW_BoW_2026-07-10_12-19-...
9,2026-07-10 12:27:24,Random Forest - Word2Vec,Word2Vec,"Random Forest model with word2vec(100 dims, sg=1)",1.000000,0.960946,0.970648,0.95075,0.960596,"[[3874, 115], [197, 3803]]",models\Random_Forest_-_Word2Vec_Word2Vec_2026-...


In [ ]:
# --- Bar plot comparing metrics across models ---
metrics_to_plot = ['train_accuracy', 'test_accuracy', 'precision', 'recall', 'f1_score']
plot_df = results_df.melt(id_vars='model_name', value_vars=metrics_to_plot,
                           var_name='metric', value_name='score')

plt.figure(figsize=(12, 5))
sns.barplot(data=plot_df, x='model_name', y='score', hue='metric')
plt.title('Model Comparison Across Metrics (Word2Vec)')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


: 

In [ ]:
# --- F1 score by model ---
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='model_name', y='f1_score')
plt.title('F1 Score by Model (Word2Vec)')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


: 

In [ ]:
# --- Confusion matrices side by side (Word2Vec models only) ---
w2v_results = results_df[results_df['method'] == 'Word2Vec'].reset_index(drop=True)

fig, axes = plt.subplots(1, len(w2v_results), figsize=(5 * len(w2v_results), 4))
if len(w2v_results) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, w2v_results.iterrows()):
    cm = np.array(row['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(row['model_name'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()


: 

In [ ]:
# --- Best model selection ---
best_row = results_df.iloc[0]
print(f"Best model based on F1 Score: {best_row['model_name']} "
      f"(F1={best_row['f1_score']:.4f}, Test Accuracy={best_row['test_accuracy']:.4f})")
print(f"Saved model file: {best_row['model_path']}")


: 

In [ ]:
# Reload the best model later, e.g. for inference:
# best_model = utils.load_model_from_file(best_row['model_path'])


: 